# Nomic Embed Multimodal - Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harpreetsahota204/nomic-embed-multimodal/blob/main/quick_start.ipynb)

A minimal example showing how to use Nomic Embed Multimodal models with FiftyOne for document understanding.


## Installation


In [ ]:
%pip install -q fiftyone colpali-engine torch transformers pillow


## Load Sample Dataset


In [ ]:
import fiftyone as fo
from fiftyone.utils.huggingface import load_from_hub

# Load a small document dataset
dataset = load_from_hub(
    "Voxel51/document-haystack-10pages",
    overwrite=True,
    max_samples=50
)

print(f"Loaded {len(dataset)} samples")


## Register Zoo Model


In [ ]:
import fiftyone.zoo as foz

# Register the remote zoo model source
foz.register_zoo_model_source(
    "https://github.com/harpreetsahota204/nomic-embed-multimodal",
    overwrite=True
)


## Compute Embeddings


In [ ]:
# Load the 3B model (faster for quick testing)
model = foz.load_zoo_model("nomic-ai/nomic-embed-multimodal-3b")

# Compute embeddings
dataset.compute_embeddings(
    model=model,
    embeddings_field="embeddings"
)

print(f"Embedding shape: {dataset.first()['embeddings'].shape}")


## Text-to-Image Search


In [ ]:
import fiftyone.brain as fob

# Build similarity index
fob.compute_similarity(
    dataset,
    model=model,
    embeddings_field="embeddings",
    brain_key="sim"
)

# Search with text query
results = dataset.sort_by_similarity(
    "invoice",
    brain_key="sim",
    k=5
)

print(f"Found {len(results)} results")


## Zero-Shot Classification

In [ ]:
# Configure model for classification
model.classes = ["invoice", "receipt", "form", "letter", "other"]
model.text_prompt = "This document is a"

# Apply zero-shot classification
dataset.apply_model(model, label_field="document_type")

# Show prediction
sample = dataset.first()
print(f"Predicted: {sample['document_type'].label}")
print(f"Confidence: {sample['document_type'].confidence:.2f}")


## Visualize Results


In [ ]:
# Launch FiftyOne App
session = fo.launch_app(results)


## Next Steps

- Try the 7B model for better accuracy: `foz.load_zoo_model("nomic-ai/nomic-embed-multimodal-7b")`
- Explore UMAP visualization: `fob.compute_visualization()`
- Detect duplicates: `fob.compute_uniqueness()`
- Check the [README](https://github.com/harpreetsahota204/nomic-embed-multimodal) for more examples
